In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np
import pandas as pd
import os
from PIL import Image

# ===== CONFIG =====
DATA_DIR = "C:/DUT/Ki 2/Edge AI/dataset/"
TRAIN_DIR = os.path.join(DATA_DIR, "train")
TEST_DIR = os.path.join(DATA_DIR, "test")

IMG_SIZE = 96
BATCH_SIZE = 32

# ===== LOAD DATA (80/20 SPLIT) =====
train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    TRAIN_DIR,
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE
)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    TRAIN_DIR,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE
)

class_names = train_ds.class_names
NUM_CLASSES = len(class_names)

# ===== OPTIMIZE PIPELINE =====
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.map(lambda x, y: (x/255.0, y)).cache().shuffle(1000).prefetch(AUTOTUNE)
val_ds = val_ds.map(lambda x, y: (x/255.0, y)).cache().prefetch(AUTOTUNE)

# ===== DATA AUGMENT =====
data_aug = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

# ===== MBConv Block =====
def mbconv(x, out_channels, expansion=4, stride=1):
    in_channels = x.shape[-1]

    y = layers.Conv2D(in_channels * expansion, 1, padding='same', use_bias=False)(x)
    y = layers.BatchNormalization()(y)
    y = layers.ReLU()(y)

    y = layers.DepthwiseConv2D(3, strides=stride, padding='same', use_bias=False)(y)
    y = layers.BatchNormalization()(y)
    y = layers.ReLU()(y)

    y = layers.Conv2D(out_channels, 1, padding='same', use_bias=False)(y)
    y = layers.BatchNormalization()(y)

    if stride == 1 and in_channels == out_channels:
        y = layers.Add()([x, y])

    return y

# ===== MODEL =====
def build_model():
    inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = data_aug(inputs)

    x = layers.Conv2D(16, 3, strides=2, padding='same')(x)

    x = mbconv(x, 16)
    x = mbconv(x, 24, stride=2)
    x = mbconv(x, 24)

    x = mbconv(x, 32, stride=2)
    x = mbconv(x, 32)

    x = mbconv(x, 64, stride=2)
    x = mbconv(x, 64)

    x = layers.GlobalAveragePooling2D()(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

    return models.Model(inputs, outputs)

model = build_model()

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

callbacks = [
    tf.keras.callbacks.ReduceLROnPlateau(patience=3),
    tf.keras.callbacks.EarlyStopping(patience=6, restore_best_weights=True)
]

# ===== TRAIN =====
print("Training...")
model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30,
    callbacks=callbacks
)

# ===== SAVE MODEL =====
model.save("model.h5")

# ===== CONVERT TFLITE =====
print("Converting to TFLite...")
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]

tflite_model = converter.convert()

with open("model.tflite", "wb") as f:
    f.write(tflite_model)

# ===== LOAD TFLITE =====
interpreter = tf.lite.Interpreter(model_path="model.tflite")
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

# ===== PREPROCESS =====
def preprocess(img_path):
    img = Image.open(img_path).convert("RGB")
    img = img.resize((IMG_SIZE, IMG_SIZE))
    img = np.array(img) / 255.0
    return img.astype(np.float32)

# ===== PREDICT =====
print("Predicting...")
ids = []
preds = []

for file in os.listdir(TEST_DIR):
    path = os.path.join(TEST_DIR, file)

    img = preprocess(path)
    img = np.expand_dims(img, axis=0)

    interpreter.set_tensor(input_details[0]['index'], img)
    interpreter.invoke()

    output = interpreter.get_tensor(output_details[0]['index'])
    pred = np.argmax(output)

    ids.append(file)
    preds.append(pred)

# ===== SAVE SUBMISSION =====
df = pd.DataFrame({
    "id": ids,
    "label": preds
})

df.to_csv("submission.csv", index=False)

print("Done! submission.csv created.")